# Modelo Predictivo de Retrasos y Cancelaciones de Vuelos
### Machine Learning con scikit-learn y Arquitectura Medallón

---

## 🎯 Objetivo de Negocio y Machine Learning
El objetivo es predecir si un vuelo comercial programado sufrirá una disrupción severa, definida como:
- **Retraso en la salida superior o igual a 15 minutos** (`Dep_Delay >= 15`), o
- **Cancelación total del vuelo** (`Cancelled == 1`).

Esta variable objetivo binaria (`is_delayed_or_cancelled`) permite a las aerolíneas y operadores aeroportuarios anticipar riesgos operativos, reasignar tripulaciones y mitigar el impacto para los pasajeros.

### 🧪 Metodología:
1. **Carga y Verificación:** Ingesta del dataset de características desde la **Capa Oro** (`ml_features_dataset.parquet`).
2. **EDA Focalizado:** Análisis del desbalance de clases y correlación con variables meteorológicas y temporales.
3. **Feature Engineering & Preprocesamiento:**
   - Variables numéricas (temperatura, lluvia, nieve, viento, presión atmosférica, edad del avión).
   - Variables categóricas (aerolínea, aeropuerto agrupado por frecuencia, franja horaria, día de la semana, mes).
   - `ColumnTransformer` + `Pipeline` para garantizar cero fuga de datos (*data leakage*).
4. **Comparación de Modelos:**
   - **Baseline:** Regresión Logística con pesos balanceados (`class_weight='balanced'`).
   - **Challenger:** `HistGradientBoostingClassifier` (árboles de decisión potenciados con soporte nativo de histogramas rápidos).
5. **Evaluación Rigurosa:** Curvas ROC-AUC, Precision-Recall, F1-Score, Matriz de Confusión y Feature Importance por permutación.
6. **Serialización:** Exportación del mejor pipeline entrenado a la carpeta `models/` para su consumo en la aplicación Streamlit.


In [ ]:
import os
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)
from sklearn.inspection import permutation_importance

# Configuración de estilo gráfico
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110

# Configurar rutas del proyecto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import config
config.ensure_dirs()

print("✓ Librerías importadas y entorno inicializado.")


## 1. Carga de Datos desde la Capa Oro
Cargamos la tabla de características preparada durante el pipeline ETL.


In [ ]:
ml_features_path = config.GOLD_DIR / "ml_features_dataset.parquet"

if ml_features_path.exists():
    print(f"Cargando dataset desde Capa Oro: {ml_features_path.name}")
    df = pd.read_parquet(ml_features_path)
else:
    print("⚠️ El dataset consolidado en Oro no fue encontrado. Verifique la ejecución de 01_etl_pipeline.ipynb.")
    silver_files = list(config.SILVER_FLIGHTS.glob("*.parquet"))
    if silver_files:
        print("Cargando muestra de contingencia desde Capa Plata...")
        df = pd.read_parquet(silver_files[0]).sample(n=25000, random_state=42)
        df['is_delayed_or_cancelled'] = ((df['Dep_Delay'] >= 15) | (df['Cancelled'] == 1)).astype('int8')
        df['Month'] = pd.to_datetime(df['FlightDate']).dt.month.astype('int8')
        df['Is_Weekend'] = df['Day_Of_Week'].isin([6, 7]).astype('int8')
        for w in ['tavg', 'prcp', 'snow', 'wspd', 'pres']:
            df[w] = 0.0

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas.")
df.head(5)


## 2. Análisis Exploratorio de Datos (EDA) y Desbalance de Clases
Examinamos la distribución de la variable objetivo y el impacto de los factores climáticos.


In [ ]:
target_counts = df['is_delayed_or_cancelled'].value_counts()
target_pct = df['is_delayed_or_cancelled'].value_counts(normalize=True) * 100

print("=" * 60)
print("DISTRIBUCIÓN DE LA VARIABLE OBJETIVO:")
for val, count, pct in zip(target_counts.index, target_counts.values, target_pct.values):
    label = "Disrupción (Retraso >=15m / Cancelado)" if val == 1 else "A Tiempo / Retraso Menor (<15m)"
    print(f"  Clase {val} ({label:<35}): {count:>7,} ({pct:>5.2f}%)")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. Gráfico de barras de desbalance
sns.barplot(x=['A Tiempo (0)', 'Disrupción (1)'], y=target_counts.values, palette=['#3b82f6', '#ef4444'], ax=axes[0])
axes[0].set_title("Proporción de Clases")
axes[0].set_ylabel("Cantidad de Vuelos")

# 2. Tasa de retraso/cancelación por mes
monthly_rate = df.groupby('Month')['is_delayed_or_cancelled'].mean() * 100
sns.lineplot(x=monthly_rate.index, y=monthly_rate.values, marker='o', color='#0284c7', linewidth=2, ax=axes[1])
axes[1].set_title("Tasa de Disrupción por Mes (%)")
axes[1].set_xlabel("Mes del Año")
axes[1].set_ylabel("% Disrupción")

# 3. Impacto de velocidad de viento
df['wind_bin'] = pd.qcut(df['wspd'], q=4, duplicates='drop')
wind_rate = df.groupby('wind_bin', observed=False)['is_delayed_or_cancelled'].mean() * 100
sns.barplot(x=wind_rate.index.astype(str), y=wind_rate.values, palette='Blues', ax=axes[2])
axes[2].set_title("Tasa de Disrupción vs Cuartiles de Viento (km/h)")
axes[2].set_xlabel("Rango de Velocidad del Viento")
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 3. Definición del Espacio de Características y Pipeline de Preprocesamiento
Configuramos transformadores independientes para variables numéricas y categóricas:
- **Numéricas:** Imputación de edad del avión y escalamiento estándar (`StandardScaler`).
- **Categóricas:** Agrupamiento de aeropuertos menores en categoría `'OTHER'` para evitar explosión dimensional y codificación One-Hot (`OneHotEncoder`).


In [ ]:
# 1. Seleccionar columnas predictoras
numeric_features = ['tavg', 'prcp', 'snow', 'wspd', 'pres']

if 'Aircraft_age' in df.columns:
    df['Aircraft_age'] = pd.to_numeric(df['Aircraft_age'], errors='coerce')
    df['Aircraft_age'] = df['Aircraft_age'].fillna(df['Aircraft_age'].median())
    numeric_features.append('Aircraft_age')

# Agrupar aeropuertos de baja frecuencia en 'OTHER'
top_30_airports = set(df['dep_airport'].value_counts().nlargest(30).index)
df['dep_airport_grouped'] = df['dep_airport'].apply(lambda x: x if x in top_30_airports else 'OTHER')

categorical_features = ['Day_Of_Week', 'Month', 'Is_Weekend', 'DepTime_label', 'airline', 'dep_airport_grouped']

feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y = df['is_delayed_or_cancelled'].copy()

# 2. Pipeline de preprocesamiento con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# 3. Partición Train / Test estratificada (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Variables seleccionadas: {len(feature_cols)} ({len(numeric_features)} numéricas, {len(categorical_features)} categóricas)")
print(f"Dimensiones de entrenamiento: {X_train.shape[0]:,} muestras")
print(f"Dimensiones de prueba:        {X_test.shape[0]:,} muestras")


## 4. Entrenamiento y Comparación de Modelos
Entrenamos dos clasificadores:
1. **Regresión Logística (Baseline)** con penalización regularizada y `class_weight='balanced'`.
2. **HistGradientBoostingClassifier (Challenger)** basado en árboles potenciados por gradiente con discretización en histogramas.


In [ ]:
# 4.1 Baseline: Regresión Logística
print("Entrenando Modelo 1: Regresión Logística (Baseline)...")
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)
y_proba_lr = pipe_lr.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, y_proba_lr)
f1_lr = f1_score(y_test, y_pred_lr)
print(f"-> Regresión Logística: ROC-AUC = {auc_lr:.4f} | F1 = {f1_lr:.4f}")

# 4.2 Challenger: HistGradientBoostingClassifier
print("
Entrenando Modelo 2: HistGradientBoostingClassifier (Árboles Potenciados)...")
pipe_hgb = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.08,
        max_depth=8,
        class_weight='balanced',
        random_state=42
    ))
])
pipe_hgb.fit(X_train, y_train)

y_pred_hgb = pipe_hgb.predict(X_test)
y_proba_hgb = pipe_hgb.predict_proba(X_test)[:, 1]
auc_hgb = roc_auc_score(y_test, y_proba_hgb)
f1_hgb = f1_score(y_test, y_pred_hgb)
print(f"-> HistGradientBoosting: ROC-AUC = {auc_hgb:.4f} | F1 = {f1_hgb:.4f}")


## 5. Evaluación Exhaustiva de Desempeño
Seleccionamos el mejor modelo y analizamos en detalle su matriz de confusión, reporte de clasificación y curvas ROC.


In [ ]:
best_model = pipe_hgb if auc_hgb >= auc_lr else pipe_lr
best_name = "HistGradientBoosting" if auc_hgb >= auc_lr else "Logistic Regression"
y_pred_best = y_pred_hgb if auc_hgb >= auc_lr else y_pred_lr
y_proba_best = y_proba_hgb if auc_hgb >= auc_lr else y_proba_lr

print("=" * 65)
print(f"REPORTE DE CLASIFICACIÓN FINAL - MODELO GANADOR: {best_name}")
print("=" * 65)
print(classification_report(y_test, y_pred_best, target_names=['A Tiempo (0)', 'Disrupción (1)']))

# Gráficos de evaluación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: A Tiempo', 'Pred: Disrupción'],
            yticklabels=['Real: A Tiempo', 'Real: Disrupción'])
axes[0].set_title(f"Matriz de Confusión ({best_name})")

# Curvas ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_hgb, tpr_hgb, _ = roc_curve(y_test, y_proba_hgb)

axes[1].plot(fpr_lr, tpr_lr, label=f"LogReg (AUC = {auc_lr:.3f})", color='#64748b', linestyle='--')
axes[1].plot(fpr_hgb, tpr_hgb, label=f"HistGradBoost (AUC = {auc_hgb:.3f})", color='#0ea5e9', linewidth=2.5)
axes[1].plot([0, 1], [0, 1], 'k:', alpha=0.4)
axes[1].set_title("Curva ROC Comparativa")
axes[1].set_xlabel("Tasa de Falsos Positivos (FPR)")
axes[1].set_ylabel("Tasa de Verdaderos Positivos (TPR)")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## 6. Importancia de Características (Permutation Importance)
Calculamos la importancia de cada variable evaluando el impacto del desorden aleatorio en el conjunto de prueba.


In [ ]:
print("Calculando importancia de características mediante permutación en test set...")
perm_result = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)

top_indices = perm_result.importances_mean.argsort()[::-1][:10]

plt.figure(figsize=(10, 5))
plt.barh(
    [X_test.columns[i] for i in top_indices][::-1],
    perm_result.importances_mean[top_indices][::-1],
    color='#0284c7'
)
plt.title("Top 10 Características Clave para la Predicción de Retrasos y Cancelaciones")
plt.xlabel("Pérdida de Rendimiento al Permutar (Mean Accuracy Drop)")
plt.tight_layout()
plt.show()


## 7. Serialización del Modelo en Producción
Guardamos el pipeline completo junto con metadatos clave en `models/flight_delay_predictor.joblib`.


In [ ]:
model_artifact_path = config.MODELS_DIR / "flight_delay_predictor.joblib"

payload = {
    'model_name': best_name,
    'pipeline': best_model,
    'features': feature_cols,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'metrics': {
        'roc_auc': max(auc_lr, auc_hgb),
        'f1_score': max(f1_lr, f1_hgb)
    },
    'target_description': 'is_delayed_or_cancelled: 1 si Dep_Delay >= 15 o Cancelled == 1, 0 en caso contrario'
}

joblib.dump(payload, model_artifact_path)
print("=" * 65)
print(f"✓ Modelo serializado con éxito en: {model_artifact_path}")
print(f"• Algoritmo: {best_name}")
print(f"• ROC-AUC:    {payload['metrics']['roc_auc']:.4f}")
print(f"• F1-Score:   {payload['metrics']['f1_score']:.4f}")
print("=" * 65)
